# **Quy trình huấn luyện**

Quy trình huấn luyện seq2seq cho tóm tắt văn bản tiếng Việt

### Thiết bị hỗ trợ
- Ưu tiên **GPU CUDA** và dùng `fp16` để giảm bộ nhớ
- Nếu không có GPU, notebook tự chuyển sang **CPU** với `fp32`
- Khi chạy local, notebook tự bật smoke test và chạy 1 bước cho cả hai giai đoạn
- Kết quả smoke test chỉ dùng để kiểm tra luồng, không dùng để báo cáo chất lượng mô hình

In [ ]:
from __future__ import annotations
import os
from pathlib import Path

is_kaggle = Path("/kaggle/working").exists()
is_colab = "COLAB_RELEASE_TAG" in os.environ
is_local = not is_kaggle and not is_colab
smoke_test = is_local

if is_local and not Path("pyproject.toml").exists():
    parent_dir = Path.cwd().parent
    if (parent_dir / "pyproject.toml").exists():
        os.chdir(parent_dir)

# Giảm phân mảnh bộ cấp phát CUDA, phải đặt trước khi khởi tạo mô hình hoặc GPU
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print(f"Chế độ chạy: {'SMOKE TEST LOCAL' if smoke_test else 'HUẤN LUYỆN ĐẦY ĐỦ'}")


In [5]:
from __future__ import annotations
import os

REPO_URL = "https://github.com/dungcony/sumarization.git"

# Local dùng trực tiếp mã nguồn hiện tại, Kaggle hoặc Colab mới cần tải kho mã
if is_local:
    print(f"Dùng mã nguồn local tại: {Path.cwd()}")
elif os.path.exists(".git") and "sumarization" in os.getcwd():
    print("Đang cập nhật code mới nhất...")
    !git pull
else:
    print("Đang tải mã nguồn...")
    !git clone {REPO_URL} sumarization
    %cd sumarization

Đang cập nhật code mới nhất...


302.98s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Already up to date.


In [6]:
%pip install -e . 2>&1 | tail -5

311.80s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  Attempting uninstall: vn-summarization
    Found existing installation: vn-summarization 1.0.0
    Uninstalling vn-summarization-1.0.0:
      Successfully uninstalled vn-summarization-1.0.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import warnings

warnings.filterwarnings('ignore')  # Ẩn các cảnh báo không cần thiết

import time
from pathlib import Path

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

In [8]:
from src.callbacks import TrainingProgressCallback
from src.config import load_config, apply_overrides, config_to_dict
from src.data import load_and_preprocess
from src.evaluator import build_compute_metrics, evaluate_checkpoint
from src.model import (
    apply_lora,
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.trainer import train
from src.utils import (
    format_duration,
    format_number,
    save_json,
    set_seed,
    setup_logger,
    count_parameters,
)

logger = setup_logger("notebook")
print("✅ Import thành công!")

✅ Import thành công!


In [ ]:
CONFIG_FILE = "configs/vit5_base_phase_1.yaml"
config = load_config(CONFIG_FILE)

runtime_root = (
    Path("/kaggle/working")
    if is_kaggle
    else Path("/content") if is_colab else Path.cwd()
)
output_relative = Path("outputs_smoke/phase_1") if smoke_test else Path(config.training.output_dir)
output_model = runtime_root / output_relative

best_dir = output_model / "best"

# Cập nhật lại cấu hình để Trainer lưu đúng đường dẫn tuyệt đối
config = apply_overrides(config, {
    "training.output_dir": str(output_model),
})

print(f"Thư mục môi trường: {runtime_root}")
print(f"Output đang sử dụng: {output_model}")
print(f"Best checkpoint: {best_dir}")
print(config.data)

---
## 2. Kiểm tra GPU hoặc CPU

In [ ]:
import torch

is_gpu = torch.cuda.is_available()
device = torch.device("cuda" if is_gpu else "cpu")
hardware = "GPU" if is_gpu else "CPU"

print(f"Thiết bị đang dùng: {hardware}")
if is_gpu:
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")
    print(f"Số GPU: {torch.cuda.device_count()}")
else:
    print("Không tìm thấy CUDA, notebook sẽ chạy bằng CPU")

---
## 3. Cấu hình phần cứng cho GPU hoặc CPU

In [ ]:
is_gpu = torch.cuda.is_available()
hardware = "GPU" if is_gpu else "CPU"

# Ghi đè các thiết lập cho lần 1
phase_1_overrides = {
    "training.num_train_epochs": 4,
    # Độ chính xác số học và bộ tối ưu
    "training.precision": "fp16" if is_gpu else "fp32",
    # Adafactor giảm đáng kể bộ nhớ trạng thái tối ưu cho T5 trên GPU 16 GB
    "training.optim": "adafactor",

    # Thư mục lưu kết quả Lần 1
    "training.output_dir": output_model,

    # Kích thước lô dữ liệu
    "training.per_device_train_batch_size": 8 if is_gpu else 1,
    "training.per_device_eval_batch_size": 1,
    "training.gradient_accumulation_steps": 2 if is_gpu else 1,

    "training.gradient_checkpointing": False,
    # LabelSmoother tạo tensor log_softmax rất lớn và có thể gây tràn bộ nhớ tại đây
    "training.label_smoothing_factor": 0.0 if is_gpu else config.training.label_smoothing_factor,

    "training.logging_steps": 10,
}

if smoke_test:
    # Chỉ kiểm tra luồng, các chỉ số ở chế độ này không có ý nghĩa đánh giá
    local_checkpoint = Path("outputs/best")
    phase_1_overrides.update({
        "data.max_train_samples": 4,
        "data.max_eval_samples": 2,
        "data.max_source_length": 128,
        "data.max_target_length": 64,
        "training.num_train_epochs": 1,
        "training.max_steps": 1,
        "training.eval_strategy": "steps",
        "training.eval_steps": 1,
        "training.save_strategy": "steps",
        "training.save_steps": 1,
        "training.save_total_limit": 1,
        "training.logging_steps": 1,
        "generation.max_length": 64,
        "generation.min_length": 2,
        "generation.num_beams": 1,
        "generation.early_stopping": False,
    })
    if local_checkpoint.exists():
        phase_1_overrides["model.name_or_path"] = str(local_checkpoint.resolve())

config = apply_overrides(config, phase_1_overrides)

print(f"🔹 TRAIN LẦN {config.phase.name}: Học nền tảng (Parquet tổng hợp)")
print(f"Hardware:   {hardware}")
print(f"Model:      {config.model.name_or_path}")
print(f"Train data: {config.data.train_file}")
print(f"Precision:  {config.training.precision}")
print(f"Batch size: {config.training.per_device_train_batch_size}")
print(f"Eval batch: {config.training.per_device_eval_batch_size}")
print(f"LR:         {config.training.learning_rate}")
print(f"Optimizer:  {config.training.optim}")
print(f"Grad ckpt:  {config.training.gradient_checkpointing}")
print(f"Label smooth: {config.training.label_smoothing_factor}")
print(f"Output:     {config.training.output_dir}")


---
## 4. Nạp mô hình và dữ liệu

In [19]:
# Thiết lập hạt giống ngẫu nhiên
set_seed(config.training.seed)

# Nạp bộ mã hóa
tokenizer = load_tokenizer(config.model)
print(f"✅ Tokenizer: vocab_size={tokenizer.vocab_size}")

[INFO] src.model: Đang tải T5 SentencePiece tokenizer cho: VietAI/vit5-base
[INFO] src.model: Đã tải tokenizer: vocab_size=36000, type=T5Tokenizer


✅ Tokenizer: vocab_size=36000


In [20]:
# Nạp mô hình
model = load_model(config.model, tokenizer, config.generation)

# Lưu điểm kiểm tra gradient để giảm bộ nhớ GPU
if config.training.gradient_checkpointing:
    enable_gradient_checkpointing(model)

# Đóng băng bộ mã hóa nếu cần
if config.training.freeze_encoder:
    freeze_encoder(model)

# Áp dụng LoRA nếu bật
model = apply_lora(model, config.lora)

params = count_parameters(model)
print(f"✅ Model loaded: {format_number(params['total'])} total, "
      f"{format_number(params['trainable'])} trainable ({params['trainable_percent']}%)")

[INFO] src.model: Đang tải mô hình: VietAI/vit5-base
[INFO] src.model: Đã tải mô hình: 225,950,976 tổng số tham số, 225,950,976 có thể huấn luyện (100.0%)
[INFO] src.model: LoRA bị vô hiệu hóa, sử dụng fine-tuning toàn bộ (full fine-tuning)


✅ Model loaded: 225,950,976 total, 225,950,976 trainable (100.0%)


In [21]:
# Nạp và tiền xử lý dữ liệu
datasets = load_and_preprocess(tokenizer, config.data)

print(f"✅ Train:      {len(datasets['train'])} samples")
print(f"✅ Validation: {len(datasets['validation'])} samples")
if 'test' in datasets:
    print(f"✅ Test:       {len(datasets['test'])} samples")


[INFO] src.data: Đang tải dữ liệu: train=1 file; validation=1 file; test=1 file
[INFO] src.data: Đã tải tập dữ liệu: 10775 train, 1348 validation, 1344 test
[INFO] src.data: train: đã tokenize 10775 mẫu
[INFO] src.data: validation: đã tokenize 1348 mẫu
[INFO] src.data: test: đã tokenize 1344 mẫu


✅ Train:      10775 samples
✅ Validation: 1348 samples
✅ Test:       1344 samples


---
## 5. Cấu hình bộ huấn luyện

In [22]:
tc = config.training
output_dir = Path(tc.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

# Xác định độ chính xác số học
fp16 = is_gpu and tc.precision == "fp16"

training_args = Seq2SeqTrainingArguments(
    output_dir=tc.output_dir,
    seed=tc.seed,

    # Số vòng lặp và số bước
    num_train_epochs=tc.num_train_epochs,
    max_steps=tc.max_steps,

    # Kích thước lô dữ liệu
    per_device_train_batch_size=tc.per_device_train_batch_size,
    per_device_eval_batch_size=tc.per_device_eval_batch_size,
    gradient_accumulation_steps=tc.gradient_accumulation_steps,

    # Bộ tối ưu
    learning_rate=tc.learning_rate,
    weight_decay=tc.weight_decay,
    warmup_ratio=tc.warmup_ratio,
    lr_scheduler_type=tc.lr_scheduler_type,
    optim=tc.optim,  # Thuật toán cập nhật trọng số

    # Điều chuẩn
    label_smoothing_factor=tc.label_smoothing_factor,

    # GPU dùng fp16, CPU dùng fp32
    fp16=fp16,

    # Đánh giá và lưu kết quả
    eval_strategy=tc.eval_strategy,  # Số bước học cho mỗi lần kiểm tra
    eval_steps=tc.eval_steps,
    save_strategy=tc.save_strategy,  # Số bước học cho mỗi lần lưu
    save_steps=tc.save_steps,
    save_total_limit=tc.save_total_limit,  # Số bản lưu trữ được giữ lại
    logging_strategy="steps",
    logging_steps=tc.logging_steps,
    logging_first_step=True,
    # Tắt thanh tiến trình ghi đè để hàm gọi lại bên dưới in nhật ký ra đầu ra chuẩn của Kaggle
    disable_tqdm=True,

    # Mô hình tốt nhất
    metric_for_best_model=tc.metric_for_best_model,
    greater_is_better=tc.greater_is_better,
    load_best_model_at_end=tc.load_best_model_at_end,  # Chọn bản tốt nhất để lưu

    # Sinh văn bản khi đánh giá
    predict_with_generate=True,  # Sinh bản tóm tắt trước khi chấm điểm
    generation_max_length=config.generation.max_length,  # Số mã từ tối đa được sinh khi kiểm tra

    # Ghi nhật ký bằng TensorBoard
    report_to=["tensorboard"],
    logging_dir=str(output_dir / "logs"),

)

print(f"✅ TrainingArguments đã sẵn sàng")
print(f"   fp16={fp16}")
print(f"   device: {training_args.device}")

PermissionError: [Errno 13] Permission denied: '/content'

In [ ]:
# Bộ gom dữ liệu với đệm động
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

In [ ]:
# Hàm tính các chỉ số ROUGE
compute_metrics = build_compute_metrics(tokenizer)

In [ ]:
# Theo dõi tiến trình và dừng sớm
progress_callback = TrainingProgressCallback(
    label="PHASE 1",
    log_every_steps=tc.logging_steps,
    heartbeat_seconds=60,
    log_file=output_dir / "training_progress.log",
)
callbacks = [progress_callback]
if tc.early_stopping_patience > 0:
    callbacks.append(
        EarlyStoppingCallback(
            early_stopping_patience=tc.early_stopping_patience,
        )
    )
    print(f"✅ Dừng sớm: patience={tc.early_stopping_patience}")

In [ ]:
# Khởi tạo bộ huấn luyện
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=datasets["train"],
    eval_dataset=datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=callbacks,
)

print("✅ Bộ huấn luyện đã sẵn sàng")

---
## 6. Huấn luyện 🚀

In [ ]:
print("=" * 60)
print("BẮT ĐẦU HUẤN LUYỆN")
print("=" * 60)
print(f"  Model:       {config.model.name_or_path}")
print(f"  Epochs:      {tc.num_train_epochs}")
print(f"  Batch size:  {tc.per_device_train_batch_size}")
print(f"  Grad accum:  {tc.gradient_accumulation_steps}")
print(f"  GPU count:   {training_args.n_gpu}")
print(
    f"  Effective batch: "
    f"{tc.per_device_train_batch_size * max(1, training_args.n_gpu) * tc.gradient_accumulation_steps}"
)
print(f"  LR:          {tc.learning_rate}")
print(f"  Optimizer:   {tc.optim}")
print(f"  Precision:   {tc.precision}")
print(f"  LoRA:        {config.lora.enabled}")
print(f"  Train samples: {len(datasets['train'])}")
print(f"  Log file:    {output_dir / 'training_progress.log'}")
print("  Dấu hiệu đang chạy: dòng '♥ VẪN ĐANG TRAIN' sẽ xuất hiện mỗi 60 giây")
print("=" * 60)

if is_gpu:
    torch.cuda.empty_cache()

start_time = time.time()

try:
    train_result = trainer.train(
        resume_from_checkpoint=tc.resume_from_checkpoint,
    )
finally:
    # Dừng tín hiệu duy trì cả khi người dùng ngắt hoặc quá trình huấn luyện phát sinh lỗi
    progress_callback.stop()

elapsed = time.time() - start_time
print(f"\n✅ Huấn luyện hoàn thành! Thời gian: {format_duration(elapsed)}")

---
## 7. Lưu mô hình và đánh giá

In [ ]:
# Lưu mô hình tốt nhất

print(f"Đang lưu model tốt nhất tới: {best_dir}")
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
print("✅ Đã lưu model!")

In [ ]:
# Đánh giá cuối cùng trên tập xác thực
print("Đang chạy đánh giá trên tập Validation...")
eval_results = trainer.evaluate(metric_key_prefix="eval")

print("\n" + "=" * 60)
print("KẾT QUẢ ĐÁNH GIÁ - TẬP VALIDATION")
print("=" * 60)
for key, value in sorted(eval_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
print("=" * 60)

# Đánh giá trên tập kiểm thử độc lập hoàn toàn
if 'test' in datasets:
    print("\nĐang đánh giá trên tập TEST...")
    test_results = trainer.evaluate(eval_dataset=datasets['test'], metric_key_prefix='test')
    print("\n" + "=" * 60)
    print("KẾT QUẢ ĐÁNH GIÁ - TẬP TEST (Khách quan nhất)")
    print("=" * 60)
    for k, v in sorted(test_results.items()):
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
    print("=" * 60)
else:
    print("\n⚠️ Không tìm thấy tập TEST trong dữ liệu.")
    test_results = {}


---
## 8. So sánh với mô hình gốc

Sau khi model fine-tuned đã được lưu và đánh giá, nạp lại checkpoint gốc `VietAI/vit5-base`, chấm trên đúng tập test và cùng cấu hình generation.

In [ ]:
import gc

if "test" not in datasets:
    raise ValueError("Cần tập test độc lập để so sánh model gốc và model fine-tuned.")

# Kết quả mô hình đã tinh chỉnh nằm trong test_results, giải phóng mô hình khỏi RAM/GPU
# trước khi nạp lại điểm kiểm tra gốc để tránh giữ đồng thời hai mô hình
del trainer
del model
del data_collator
gc.collect()
if is_gpu:
    torch.cuda.empty_cache()

print("=" * 72)
print("ĐÁNH GIÁ MODEL GỐC SAU KHI ĐÃ HOÀN TẤT FINE-TUNE")
print("=" * 72)
print(f"Model gốc: {config.model.name_or_path}")
print(f"Model fine-tuned đã lưu: {best_dir}")
print(f"Số mẫu validation dùng chung: {len(datasets['validation'])}")
print(f"Số mẫu test dùng chung: {len(datasets['test'])}")

# Nạp mới từ mã Hugging Face trong cấu hình, đây là trọng số gốc chứ không phải best_dir
baseline_model = load_model(config.model, tokenizer, config.generation)
baseline_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=baseline_model,
    padding=True,
    label_pad_token_id=-100,
)
baseline_trainer = Seq2SeqTrainer(
    model=baseline_model,
    args=training_args,
    eval_dataset=datasets["validation"],
    tokenizer=tokenizer,
    data_collator=baseline_data_collator,
    compute_metrics=compute_metrics,
)

baseline_validation_results = baseline_trainer.evaluate(
    eval_dataset=datasets["validation"],
    metric_key_prefix="baseline_eval",
)
baseline_test_results = baseline_trainer.evaluate(
    eval_dataset=datasets["test"],
    metric_key_prefix="baseline_test",
)

print("\nKẾT QUẢ MODEL GỐC TRÊN TẬP VALIDATION")
for key, value in sorted(baseline_validation_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

print("\nKẾT QUẢ MODEL GỐC TRÊN TẬP TEST")
for key, value in sorted(baseline_test_results.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

save_json(baseline_validation_results, output_dir / "baseline_validation_results.json")
save_json(baseline_test_results, output_dir / "baseline_test_results.json")
print(f"✅ Đã lưu baseline validation/test tại: {output_dir}")

del baseline_trainer
del baseline_model
del baseline_data_collator
gc.collect()

### So sánh trực tiếp trước và sau khi tinh chỉnh

In [ ]:
comparison_results = {
    "baseline_model": config.model.name_or_path,
    "finetuned_model": str(best_dir),
    "selection_metric": "validation_rougeL",
    "test_samples": len(datasets["test"]),
    "metrics": {},
}

# Dùng tập xác thực để quyết định chạy giai đoạn 2, không dùng tập kiểm thử để chọn mô hình
baseline_validation_rouge_l = baseline_validation_results["baseline_eval_rougeL"]
finetuned_validation_rouge_l = eval_results["eval_rougeL"]
validation_rouge_l_delta = finetuned_validation_rouge_l - baseline_validation_rouge_l
phase_1_better_than_baseline = validation_rouge_l_delta > 0
comparison_results["phase_2_gate"] = {
    "baseline_validation_rougeL": baseline_validation_rouge_l,
    "finetuned_validation_rougeL": finetuned_validation_rouge_l,
    "absolute_improvement": validation_rouge_l_delta,
    "phase_1_better_than_baseline": phase_1_better_than_baseline,
}

print("\nĐIỀU KIỆN CHẠY PHASE 2 — VALIDATION ROUGE-L")
print(f"  Model gốc:       {baseline_validation_rouge_l:.2f}")
print(f"  Sau phase 1:     {finetuned_validation_rouge_l:.2f}")
print(f"  Chênh lệch:      {validation_rouge_l_delta:+.2f}")
print(f"  Chạy phase 2:    {phase_1_better_than_baseline}")

print("\n" + "=" * 72)
print("SO SÁNH MODEL GỐC VÀ MODEL SAU FINE-TUNE TRÊN CÙNG TẬP TEST")
print("=" * 72)
print(f"{'Metric':<12} {'Trước FT':>12} {'Sau FT':>12} {'Chênh lệch':>14}")
print("-" * 72)

for metric in ("rouge1", "rouge2", "rougeL"):
    baseline_value = baseline_test_results[f"baseline_test_{metric}"]
    finetuned_value = test_results[f"test_{metric}"]
    delta = finetuned_value - baseline_value
    comparison_results["metrics"][metric] = {
        "before_finetune": baseline_value,
        "after_finetune": finetuned_value,
        "absolute_improvement": delta,
    }
    print(f"{metric:<12} {baseline_value:>12.2f} {finetuned_value:>12.2f} {delta:>+14.2f}")

rouge_l_delta = comparison_results["metrics"]["rougeL"]["absolute_improvement"]
print("-" * 72)
if rouge_l_delta > 0:
    print(f"✅ Fine-tune cải thiện ROUGE-L {rouge_l_delta:+.2f} điểm trên tập test.")
elif rouge_l_delta < 0:
    print(f"⚠️ Fine-tune làm ROUGE-L giảm {rouge_l_delta:.2f} điểm trên tập test.")
else:
    print("ℹ️ ROUGE-L không thay đổi sau fine-tune.")

print("Lưu ý: ROUGE tăng là bằng chứng định lượng; vẫn nên đọc thủ công một số summary để kiểm tra tính đúng sự thật.")

In [ ]:
# Lưu chỉ số và cấu hình
train_metrics = train_result.metrics
train_metrics["train_runtime_formatted"] = format_duration(
    train_metrics.get("train_runtime", 0)
)

save_json(train_metrics, output_dir / "train_results.json")
save_json(eval_results, output_dir / "eval_results.json")
save_json(baseline_validation_results, output_dir / "baseline_validation_results.json")
save_json(baseline_test_results, output_dir / "baseline_test_results.json")
if test_results:
    save_json(test_results, output_dir / "test_results.json")
    save_json(comparison_results, output_dir / "before_after_comparison.json")
save_json(config_to_dict(config), output_dir / "resolved_config.json")

print("✅ Đã lưu tất cả metrics!")
print(f"   📁 {output_dir}")


---
## 9. Tự động huấn luyện giai đoạn 2

Khi huấn luyện đầy đủ, giai đoạn 2 chỉ chạy nếu ROUGE-L xác thực của giai đoạn 1 cao hơn mô hình gốc. Ở smoke test local, giai đoạn 2 luôn chạy 1 bước để kiểm tra toàn bộ luồng.

In [ ]:
PHASE_2_CONFIG_FILE = "configs/vit5_base_phase_2.yaml"

phase_2_completed = False
final_best_dir = best_dir
final_config = config
run_phase_2 = phase_1_better_than_baseline or smoke_test
phase_2_summary = {
    "triggered": bool(run_phase_2),
    "smoke_test": bool(smoke_test),
    "gate_passed": bool(phase_1_better_than_baseline),
    "gate_metric": "validation_rougeL",
    "phase_1_improvement": validation_rouge_l_delta,
    "phase_1_checkpoint": str(best_dir),
}

if not run_phase_2:
    print("⏭️ BỎ QUA PHASE 2")
    print(
        f"Phase 1 chưa vượt model gốc trên validation ROUGE-L "
        f"({validation_rouge_l_delta:+.2f})."
    )
    phase_2_summary["status"] = "skipped"
    save_json(phase_2_summary, output_dir / "phase_2_gate.json")


In [ ]:
# Chuẩn bị cấu hình giai đoạn 2
if run_phase_2:
    print("=" * 72)
    if smoke_test and not phase_1_better_than_baseline:
        print("SMOKE TEST — CHẠY GIAI ĐOẠN 2 ĐỂ KIỂM TRA LUỒNG")
    else:
        print("GIAI ĐOẠN 1 TỐT HƠN MÔ HÌNH GỐC — BẮT ĐẦU GIAI ĐOẠN 2")
    print("=" * 72)

    phase_2_config = load_config(PHASE_2_CONFIG_FILE)
    phase_2_runtime_root = (
        Path("/kaggle/working")
        if is_kaggle
        else Path("/content") if is_colab else Path.cwd()
    )
    phase_2_relative = Path("outputs_smoke/phase_2") if smoke_test else Path(phase_2_config.training.output_dir)
    phase_2_output_dir = phase_2_runtime_root / phase_2_relative
    phase_2_best_dir = phase_2_output_dir / "best"

    phase_2_overrides = {
        # Bắt buộc tiếp tục từ điểm kiểm tra tốt nhất của giai đoạn 1
        "model.name_or_path": str(best_dir),
        "training.output_dir": str(phase_2_output_dir),
        "training.precision": "fp16" if is_gpu else "fp32",
        "training.optim": "adafactor" if is_gpu else "adamw_torch",
        "training.per_device_train_batch_size": 1,
        "training.per_device_eval_batch_size": 1,
        "training.gradient_accumulation_steps": 1 if smoke_test else 8 if is_gpu else 1,
        "training.gradient_checkpointing": bool(is_gpu),
        "training.label_smoothing_factor": 0.0 if is_gpu else phase_2_config.training.label_smoothing_factor,
        # GPU dùng ít nhánh tìm kiếm hơn để tránh tràn bộ nhớ trong model.generate
        "generation.num_beams": 2,
    }

    if smoke_test:
        phase_2_overrides.update({
            "data.max_train_samples": 4,
            "data.max_eval_samples": 2,
            "data.max_source_length": 128,
            "data.max_target_length": 64,
            "training.num_train_epochs": 1,
            "training.max_steps": 1,
            "training.eval_strategy": "steps",
            "training.eval_steps": 1,
            "training.save_strategy": "steps",
            "training.save_steps": 1,
            "training.save_total_limit": 1,
            "training.logging_steps": 1,
            "training.early_stopping_patience": 0,
            "generation.max_length": 64,
            "generation.min_length": 2,
            "generation.num_beams": 1,
            "generation.early_stopping": False,
        })

    phase_2_config = apply_overrides(phase_2_config, phase_2_overrides)

    print(f"Nguồn model:  {phase_2_config.model.name_or_path}")
    print(f"Train data:   {phase_2_config.data.train_file}")
    print(f"Output:       {phase_2_output_dir}")
    print(f"Hardware:     {hardware}")
    print(f"Precision:    {phase_2_config.training.precision}")
    print(f"Epochs:       {phase_2_config.training.num_train_epochs}")
    print(f"Train batch:  {phase_2_config.training.per_device_train_batch_size}")
    print(f"Eval batch:   {phase_2_config.training.per_device_eval_batch_size}")
    print(f"Grad accum:   {phase_2_config.training.gradient_accumulation_steps}")
    print(f"Grad ckpt:    {phase_2_config.training.gradient_checkpointing}")
    print(f"Label smooth: {phase_2_config.training.label_smoothing_factor}")

In [ ]:
# Huấn luyện trên train và chọn checkpoint bằng validation
if run_phase_2:
    phase_2_eval_results = train(phase_2_config)

    # Tập test chỉ được chấm sau khi đã chọn xong checkpoint
    phase_2_test_results = evaluate_checkpoint(
        model_path=phase_2_best_dir,
        config=phase_2_config,
        output_dir=phase_2_output_dir,
        export_predictions=True,
        split="test",
    )

In [ ]:
# Cập nhật trạng thái và lưu kết quả giai đoạn 2
if run_phase_2:
    phase_2_completed = True
    final_best_dir = phase_2_best_dir
    final_config = phase_2_config
    phase_2_summary.update({
        "status": "completed",
        "phase_2_checkpoint": str(phase_2_best_dir),
        "phase_2_eval_rougeL": phase_2_eval_results.get("eval_rougeL"),
        "phase_2_test_rougeL": phase_2_test_results.get("test_rougeL"),
    })
    save_json(phase_2_summary, output_dir / "phase_2_gate.json")

    print("\n" + "=" * 72)
    print("KẾT QUẢ GIAI ĐOẠN 2")
    print("=" * 72)
    print(f"  Điểm kiểm tra tốt nhất: {phase_2_best_dir}")
    print(f"  Validation ROUGE-L: {phase_2_eval_results.get('eval_rougeL')}")
    print(f"  Test ROUGE-L:       {phase_2_test_results.get('test_rougeL')}")

    gc.collect()

---
## 10. Kiểm tra thử mô hình

In [ ]:
from src.predict import summarize

test_text = """Thủ tướng Chính phủ vừa phê duyệt đề án phát triển ứng dụng 
trí tuệ nhân tạo tại Việt Nam giai đoạn 2025-2030. Theo đó, Việt Nam đặt 
mục tiêu trở thành một trong những trung tâm đổi mới sáng tạo về AI trong 
khu vực ASEAN. Đề án tập trung vào 5 lĩnh vực ưu tiên gồm y tế, giáo dục, 
nông nghiệp, giao thông và sản xuất công nghiệp."""

summary = summarize(
    text=test_text,
    model_path=str(final_best_dir),
    config=final_config,
)

print(f"🔎 Checkpoint inference: {final_best_dir}")
print(f"   Phase 2 completed: {phase_2_completed}")
print("📄 Bài gốc:")
print(test_text.strip())
print("\n📝 Tóm tắt:")
print(summary)

if smoke_test:
    print("\n✅ SMOKE TEST LOCAL HOÀN TẤT TOÀN BỘ LUỒNG")
    print(f"Kết quả tạm được lưu tại: {runtime_root / 'outputs_smoke'}")